In [1]:
import pandas as pd
import glob
import os
import re

datasets = ['tax', 'hospital']

In [ ]:
# Read all mapping GT CSV files
csv_files = glob.glob('mapping_gt_results/llm_mapping_gt_*.csv')

# Load all dataframes
dfs = []
for csv_file in csv_files:
    df = pd.read_csv(csv_file)
    dfs.append(df)

# Combine all dataframes
df_combined = pd.concat(dfs, ignore_index=True)

# Parse variant_name to extract model and prompt_key
# Format: "Model DIRECT (prompt_key)"
def parse_variant_name(variant_name):
    if pd.isna(variant_name) or variant_name == 'N/A':
        return None, None
    
    # Match pattern: "Model DIRECT (prompt_key)"
    match = re.match(r'^(.+?)\s+DIRECT\s+\((.+?)\)$', variant_name)
    if match:
        model = match.group(1).strip()
        prompt_key = match.group(2).strip()
        return model, prompt_key
    return None, None

# Parse and add columns
parsed = df_combined['variant_name'].apply(parse_variant_name)
df_combined['model'] = [p[0] for p in parsed]
df_combined['prompt_key'] = [p[1] for p in parsed]

# Filter out rows where parsing failed
df_combined = df_combined[df_combined['model'].notna()].copy()

print(f"Total rows: {len(df_combined)}")
print(f"Datasets: {sorted(df_combined['dataset'].unique())}")
print(f"Models: {sorted(df_combined['model'].unique())}")
print(f"Prompt keys: {sorted(df_combined['prompt_key'].unique())}")
print()


def generate_latex_table_for_dataset(df, dataset_name):
    """
    Generate a LaTeX booktabs table for mapping results on a specific dataset.
    """
    # Filter for specific dataset and sort
    df = df[df['dataset'] == dataset_name].copy()
    df = df.sort_values(['model', 'prompt_key']).reset_index(drop=True)

    # Simplify model names for display
    model_display_names = {
        'GPT-5-mini': 'GPT-5-mini',
        'GPT-5': 'GPT-5',
        'Mistral-small': 'Mistral-Small',
        'Mistral-large': 'Mistral-Large',
        'Llama-3.3-70b': 'Llama-3.3-70b'
    }

    # Prompt key display names
    prompt_display_names = {
        'zero_shot': 'Zero-shot',
        'zero_shot_with_table': 'Zero-shot+Table',
        'with_examples': 'Few-shot',
        'with_examples_and_table': 'Few-shot+Table'
    }

    caption = (
        f"Mapping results on the {dataset_name.capitalize()} dataset. "
        f"Constraint-level: exact match precision/recall/F1 for individual constraints. "
        f"Pairwise Syntactic Jaccard: predicate similarity between constraint pairs. "
        f"Pairwise Violation: exact match and Jaccard for violation pairs. "
        f"Aggregate Violation: overall precision/recall/F1 for all violations."
    )

    latex = []
    latex.append(r'\begin{table*}[htbp]')
    latex.append(r'\centering')
    latex.append(r'\small')
    latex.append(f'\\caption{{{caption}}}')
    latex.append(f'\\label{{tab:mapping_results_{dataset_name}}}')
    latex.append(r'\begin{tabular}{@{}llccccccccc@{}}')
    latex.append(r'\toprule')
    latex.append(r' & & \multicolumn{3}{c}{\textbf{Constraint}} & \textbf{PW Synt} & \multicolumn{2}{c}{\textbf{PW Violation}} & \multicolumn{3}{c}{\textbf{Agg Violation}} \\')
    latex.append(r'\cmidrule(lr){3-5} \cmidrule(lr){6-6} \cmidrule(lr){7-8} \cmidrule(lr){9-11}')
    latex.append(r'\textbf{Model} & \textbf{Prompt} & \textbf{Prec} & \textbf{Rec} & \textbf{F1} & \textbf{Pred Jacc} & \textbf{Match} & \textbf{Jacc} & \textbf{Prec} & \textbf{Rec} & \textbf{F1} \\')
    latex.append(r'\midrule')

    prev_model = None
    model_rows = {}

    # First pass: count rows per model
    for _, row in df.iterrows():
        model = row['model']
        model_rows[model] = model_rows.get(model, 0) + 1

    # Second pass: generate table rows
    model_first_row = {}

    for _, row in df.iterrows():
        model = row['model']
        prompt_key = row['prompt_key']
        
        # Constraint-level metrics
        const_prec = f"{row['constraint_exact_precision']:.2f}"
        const_rec = f"{row['constraint_exact_recall']:.2f}"
        const_f1 = f"{row['constraint_exact_f1']:.2f}"
        
        # Pairwise syntactic
        pw_synt_jacc = f"{row['pairwise_syntactic_predicate_jaccard']:.2f}"
        
        # Pairwise violation metrics
        pw_viol_match = f"{row['pairwise_violation_exact_match']:.2f}"
        pw_viol_jacc = f"{row['pairwise_violation_jaccard']:.2f}"
        
        # Aggregate violation metrics
        agg_viol_prec = f"{row['aggregate_violation_precision']:.2f}"
        agg_viol_rec = f"{row['aggregate_violation_recall']:.2f}"
        agg_viol_f1 = f"{row['aggregate_violation_f1']:.2f}"

        model_name = model_display_names.get(model, model)
        prompt_name = prompt_display_names.get(prompt_key, prompt_key)

        # Light rule between models (but not at the start)
        if prev_model is not None and model != prev_model:
            latex.append(r'\cmidrule(lr){1-11}')

        # Model multirow
        if model not in model_first_row:
            model_cell = f"\\multirow{{{model_rows[model]}}}{{*}}{{{model_name}}}"
            model_first_row[model] = True
        else:
            model_cell = ""

        latex.append(f"{model_cell} & {prompt_name} & {const_prec} & {const_rec} & {const_f1} & {pw_synt_jacc} & {pw_viol_match} & {pw_viol_jacc} & {agg_viol_prec} & {agg_viol_rec} & {agg_viol_f1} \\\\")
        prev_model = model

    latex.append(r'\bottomrule')
    latex.append(r'\end{tabular}')
    latex.append(r'\end{table*}')

    return '\n'.join(latex)


# Generate one table per dataset
datasets = df_combined['dataset'].unique()
print("Generating mapping tables for each dataset...")
print("="*80)

for dataset in sorted(datasets):
    latex_table = generate_latex_table_for_dataset(df_combined, dataset)

    # Save to file with dataset-specific name
    output_dir = 'latex_tables'
    os.makedirs(output_dir, exist_ok=True)
    output_file = f'{output_dir}/mapping_results_{dataset}.tex'
    with open(output_file, 'w') as f:
        f.write(latex_table)

    print(f"\n{dataset.upper()} DATASET:")
    print(latex_table)
    print(f"\nSaved to: {output_file}")
    print("="*80)

print("\nAll tables generated successfully!")
print("Required package: \\usepackage{booktabs, multirow}")

Total rows: 53
Datasets: ['hospital', 'restaurant', 'tax']
Models: ['GPT-5', 'GPT-5-mini', 'Llama-3.3-70b', 'Mistral-large', 'Mistral-small']
Prompt keys: ['with_examples', 'with_examples_and_table', 'zero_shot', 'zero_shot_with_table']

Generating mapping tables for each dataset...

HOSPITAL DATASET:
\begin{table}[htbp]
\centering
\small
\caption{Mapping results on the Hospital dataset. Metrics shown: Constraint-level (exact match), Pairwise Violation (exact match and Jaccard), and Aggregate Violation (F1 score).}
\label{tab:mapping_results_hospital}
\begin{tabular}{@{}lcccccc@{}}
\toprule
\textbf{Model} & \textbf{Prompt} & \textbf{Const F1} & \textbf{PW Viol} & \textbf{PW Viol Jacc} & \textbf{Agg Viol F1} \\
\midrule
\multirow{4}{*}{GPT-5} & Few-shot & 1.00 & 1.00 & 1.00 & 1.00 \\
 & Few-shot+Table & 1.00 & 1.00 & 1.00 & 1.00 \\
 & Zero-shot & 1.00 & 1.00 & 1.00 & 1.00 \\
 & Zero-shot+Table & 1.00 & 1.00 & 1.00 & 1.00 \\
\cmidrule(lr){1-6}
\multirow{4}{*}{GPT-5-mini} & Few-shot & 1.0